# YES Bank monthly closing-price prediction

This notebook reviews the supplied monthly OHLC data, builds a one-month-ahead regression model, and evaluates it against a persistence baseline. The project brief's rubric informs the workflow (EDA, missing/outlier review, multiple models, tuning, and evaluation); it does not override the goal of a realistic time-series evaluation.

The source CSV runs from July 2005 through November 2020. Any next-month prediction from this file is historical (December 2020), not a current-market forecast.

## 1. Setup

Run this notebook from the project directory. Install the pinned packages once if they are not already available.

In [ ]:
%pip install -r requirements.txt

from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

PROJECT = Path.cwd()
DATA = PROJECT / 'data' / 'data_YesBank_StockPrices.csv'
if not DATA.exists():
    raise FileNotFoundError('Run this notebook from the yesbank_stock_prediction project folder.')
print('Project:', PROJECT)
print('CSV:', DATA)

## 2. Explore and check data quality

Potential IQR outliers are reported rather than removed because sharp market moves can be valid observations. The correlation chart is descriptive; same-month OHLC fields are related and do not establish causality.

In [ ]:
from explore import analyze

eda = analyze(DATA)
print(json.dumps({key: eda[key] for key in ['source_rows', 'clean_rows', 'rows_removed', 'missing_by_column', 'duplicate_months', 'outlier_flags_iqr', 'ohlc_consistency_violations', 'date_range']}, indent=2))
for filename in ['price_history.png', 'monthly_returns.png', 'ohlc_correlation.png']:
    display(Image(filename=str(PROJECT / 'artifacts' / filename)))

## 3. Train and evaluate

Features available at each month-end are used to predict the following month's log return, which is converted back to a closing price. Ridge, Random Forest, and Histogram Gradient Boosting are tuned using expanding time-series folds within the first 80% of rows. The final 20% is held out chronologically and is not used for tuning. The workflow selects an ML model only if rolling-fold MAE improves on persistence by at least 5%; otherwise it uses persistence.

In [ ]:
from train import train

results = train(DATA)
print(json.dumps(results, indent=2))
display(Image(filename=str(PROJECT / 'artifacts' / 'actual_vs_predicted.png')))
display(Image(filename=str(PROJECT / 'artifacts' / 'holdout_mae_comparison.png')))
display(Image(filename=str(PROJECT / 'artifacts' / 'residuals.png')))

## 4. Findings and limitations

On the supplied holdout period, persistence has a lower MAE than all three ML candidates. The best ML candidate improved rolling-fold MAE by only about 3.2%, below the 5% selection threshold, so the selected forecast uses persistence. The dataset includes no news, fundamentals, or market index features, so it cannot attribute price changes to a specific event or explain broader market movements.

In [ ]:
prediction_rows = pd.read_csv(PROJECT / 'artifacts' / 'test_predictions.csv')
display(prediction_rows.tail(10))
print('Selected model:', results['selection']['selected_model'])
print('Best ML candidate:', results['selection']['best_ml_candidate'])
print('Cross-validation improvement over persistence:', round(100 * results['selection']['cv_improvement_fraction'], 2), '%')
print('Persistence baseline MAE:', round(results['holdout_metrics']['persistence_baseline']['mae'], 2))
print('Ridge MAE:', round(results['holdout_metrics']['ridge']['mae'], 2))
print('Random Forest MAE:', round(results['holdout_metrics']['random_forest']['mae'], 2))
print('Histogram Gradient Boosting MAE:', round(results['holdout_metrics']['hist_gradient_boosting']['mae'], 2))

## 5. Forecast after updating the data

After replacing the CSV with an updated, validated monthly series, run `python train.py` and then `python predict.py`. The selected artifact may use either the ML candidate or the persistence fallback based on time-series cross-validation. Historical performance does not guarantee future performance.